# Lesson 1: First connection

This notebook sends queries only.

Run cells in order with only one serial owner. Close any previous `ser` or controller
connection first. Stop after an error; inspect `laser.last_transaction` and run
Disconnect. Do not retry an uncertain instruction.

The controller sends byte instructions with CR/LF and waits for the full CR/LF
reply **and the following prompt** before another request can begin. This lesson
expects the instrument prompt to be enabled. It does not change echo or prompt
settings. Every reply remains raw `bytes`, including echo, CR/LF and prompt.
No automatic encoding, decoding or value interpretation is used.


## 1. Connection settings

Use lesson 0 to identify the port. Confirm model and baud on the instrument.

In [1]:
PORT = "COM5"  # Use the port identified in lesson 0, e.g. "COM5"
MODEL = "V2"  # V2, V5 or V6: match the actual laser
BAUDRATE = 19200  # Match the front panel
TIMEOUT_S = 2.0

## 2. Create the controller

Construction opens nothing. Writes remain disabled.

In [2]:
from coherent_verdi import VerdiController

if PORT == "<PORT>":
    raise ValueError("Edit the connection settings in section 1 first.")
laser = VerdiController(
    PORT, model=MODEL, baudrate=BAUDRATE,
    timeout_s=TIMEOUT_S, prompt_enabled=True,
)

## 3. Open the serial port

Run this cell by itself and inspect the result before starting serial communication.
`connect()` opens the local serial port and returns `True` on success. Failures
raise an exception. It sends no instrument instructions. A successful open does
not establish that the laser is responding or ready.

Close any other serial owner first. If opening fails, stop here.

`override=True` closes an older controller connection to the same port in this
Python process before opening the new one. It cannot take ownership from another
kernel/application or a raw `ser` handle; release those explicitly. A failed
controller still requires a fresh controller object. Reopening does not establish
instrument synchronization or repair corrupted replies.

In [3]:
connected = laser.connect(override=True)
print("Serial port open:", connected)

Serial port open: True


## 4. Start serial communication

Only after inspecting the connection result, run the following cells separately.
Buffer maintenance and instrument queries belong here, outside the connection
section. An explicit purge discards host queues; it does not reset the device's
parser. No delay or purge is claimed to prove synchronization. Each query waits
for its complete reply and prompt before another can be sent.

In [4]:
laser.purge_buffers()  # Explicit host-queue cleanup once, before the first request.

## 5. Read the software version

The result includes the complete wire exchange. With echo off, V-2 firmware
6.05 has returned `b"6.05\r\nVerdi V-2>"`. Echo may precede the value.
Compare the raw bytes with the front panel; do not infer readiness from a version.

In [5]:
version_reply = laser.get(b"?SV")
print(repr(version_reply))

b'6.05\r\nVerdi V-2>'


## 6. Inspect an error

After a failed cell, this prints the last transaction without sending anything.
Run Disconnect next. An incomplete reply or missing prompt stops the session;
a new connection alone does not establish synchronization.

In [6]:
print(laser.last_transaction)

{'instruction': b'?SV', 'request': b'?SV\r\n', 'response': b'6.05\r\nVerdi V-2>', 'preamble': b'', 'payload': None, 'payload_encoding': None, 'timestamp_utc': '2026-10-26T21:45:34.421763+00:00', 'duration_s': 0.031000000017229468, 'port': 'COM5', 'model': 'V2', 'baudrate': 19200, 'timeout_s': 2.0, 'simulated': False, 'write_attempted': True, 'pending_bytes': 0, 'outcome': 'received', 'error': None}


## 7. Disconnect

Releases the port without changing laser state.

In [7]:
laser.disconnect()